# Testing and evaluating the expense checker

Model under test: `Prachi1802/ExpenseClassifier` (Qwen2.5-1.5B-Instruct fine-tuned with QLoRA, merged).

What this notebook does, in order:
1. **Held-out rows**: re-creates the exact 80/20 random split from training (seed 42) and scores the model on the 1,000 rows it never trained on.
2. **Baseline**: scores the *untrained* Qwen with the same prompt on the same rows, so you can show what fine-tuning added.
3. **Fresh cases**: 26 new expense lines with vendors and wording that are not in the dataset. Their right answers are computed from the policy table, not typed by hand.
4. **Edge cases**: awkward inputs (no currency, no amount) printed for you to look at.

**How to read the results.** (1) shows the model learned the task on the training distribution. (3) is the closer test of generalization. (2) is what justifies fine-tuning at all. Quote all three, and say what each was measured on.

Runtime: Colab, T4 GPU. Upload `expense_dataset_5000.csv` first (the same file used for training).

In [1]:
!pip install -q transformers accelerate datasets scikit-learn

## 1. Settings

In [2]:
import os, re, json, math, time, gc
import torch
import pandas as pd

MODEL_NAME = "Prachi1802/ExpenseClassifier"      # your fine-tuned, merged model on Hugging Face
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"        # the untrained model, for the baseline
CSV_PATH   = "/content/expense_dataset_5000.csv" # the same 5,000-row file used for training; change if yours is elsewhere

N_HELDOUT  = None   # None = all 1,000 held-out rows. Use 200 for a quick first run.
N_BASELINE = 200    # rows for the baseline (the untrained model is slower to run because it rambles)
BATCH      = 16     # how many expenses the model answers at once
MAX_NEW    = 220    # answer length limit: the JSON with its reason is about 100 tokens

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (use Runtime > Change runtime type > T4 GPU)")
assert os.path.exists(CSV_PATH), f"Upload the dataset to {CSV_PATH} (or change CSV_PATH)"

GPU: Tesla T4


## 2. Hugging Face login
Needed only while the model repo is private. Skip this cell if the repo is public.

In [3]:
from huggingface_hub import notebook_login
notebook_login()

## 3. The prompt
This must be **exactly** the prompt the model was trained with. A different prompt can make the answers worse for reasons unrelated to the model.

In [4]:
SYSTEM_PROMPT = """You check employee expenses against company policy.
Read the expense text and reply with only a JSON object with these keys:
amount, currency, category, policy_limit_amount, reimbursement_allowed, receipt_provided, flag, reason.

Categories: Travel, Ground Transport, Lodging, Meals, Software, Office Supplies, Training, Other.
Currencies: USD, EUR, GBP, INR, CAD.

Limits (USD / EUR / GBP / INR / CAD):
- Meals, per person: 75 / 70 / 60 / 6000 / 100
- Lodging, per night: 250 / 230 / 200 / 20000 / 340
- Ground Transport, per expense: 100 / 90 / 80 / 8000 / 135
- Travel, per expense: 800 / 750 / 650 / 64000 / 1100
- Software, per expense: 200 / 180 / 160 / 16000 / 270
- Office Supplies, per expense: 150 / 140 / 120 / 12000 / 200
- Training, per expense: 1000 / 900 / 800 / 80000 / 1350
- Other, per expense: 100 / 90 / 80 / 8000 / 135
A receipt is required above 25 USD, 25 EUR, 20 GBP, 2000 INR or 35 CAD.

Rules:
- policy_limit_amount = per-person limit x attendees for Meals, per-night limit x nights for Lodging, otherwise the limit. Use null if it cannot be worked out, and 0 if the expense is not reimbursable.
- receipt_provided is "No" only if the text says the receipt is missing.
- Not reimbursable: alcohol-only purchases, fines, upgrades, travel or rooms for family or friends, personal trips, hotel extras, personal subscriptions, gift cards, personal or home items, hobby classes, personal services or purchases.
- flag, first match wins: not_reimbursable; missing_info (amount, attendees, nights or a required receipt missing); over_limit (amount above policy_limit_amount); none.
- reimbursement_allowed is "No" only when the flag is not_reimbursable.
- Use null for anything the text does not state. reason briefly explains the flag."""

USER_TEMPLATE = "Expense: {text}"

def build_chat(text):
    # The same chat the model saw in training: system prompt + "Expense: <text>"
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(text=text)}]

## 4. Load the fine-tuned model

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype=torch.float16).eval()
print("Loaded", MODEL_NAME, "| GPU memory: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Prachi1802/ExpenseClassifier | GPU memory: 3.1 GB


## 5. Helper functions: generating and scoring
- `run_model` asks the model about a list of expense lines and returns its raw text answers.
- `score` parses each answer as JSON and compares it with the right answer field by field.

**What counts as correct.** Numbers may differ by 0.01. Text ignores upper/lower case. "Not stated" (`null`) must match "not stated". JSON keys are matched ignoring case, because the model learned a mix of cases from the training data (for example `Currency`, `Reason`).
The `reason` sentence is free text, so it is only checked for being present; the other seven fields decide `all_7_fields`.

In [6]:
from sklearn.metrics import f1_score, confusion_matrix

FIELDS7 = ["amount", "currency", "category", "policy_limit_amount",
           "reimbursement_allowed", "receipt_provided", "flag"]
NUMERIC = {"amount", "policy_limit_amount"}
FLAGS = ["none", "over_limit", "missing_info", "not_reimbursable"]

@torch.no_grad()
def run_model(model, tok, texts, batch_size=BATCH, max_new_tokens=MAX_NEW):
    # Ask the model about every expense line, a batch at a time. Greedy decoding = same answer every run.
    model.eval()
    tok.padding_side = "left"                      # pad on the left so each answer starts at the same place
    eos = tok.convert_tokens_to_ids("<|im_end|>")  # Qwen's end-of-reply token
    answers, t0 = [], time.time()
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        prompts = [tok.apply_chat_template(build_chat(t), add_generation_prompt=True, tokenize=False) for t in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             eos_token_id=eos, pad_token_id=tok.pad_token_id)
        answers += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"\r  {min(i + batch_size, len(texts))}/{len(texts)} answered, {time.time() - t0:.0f}s", end="")
    print()
    return answers

def parse_json(text):
    # Take everything from the first "{" to the last "}" (the untrained model adds extra words or code fences)
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return {str(k).lower(): v for k, v in d.items()} if isinstance(d, dict) else None

def is_blank(v):
    return v is None or (isinstance(v, float) and math.isnan(v)) or str(v).strip().lower() in ("", "null")

def same(field, pred, gold):
    # Is the model's value for one field the same as the right answer?
    if gold is None:
        return is_blank(pred)                      # right answer is "not stated": the model must also say not stated
    if is_blank(pred):
        return False
    if field in NUMERIC:
        try:
            return abs(float(pred) - float(gold)) <= 0.01
        except (TypeError, ValueError):
            return False
    return str(pred).strip().lower() == str(gold).strip().lower()

def score(items, answers):
    # items: list of {"text": ..., "gold": {field: value}}. Returns one row per expense with True/False per field.
    rows = []
    for it, ans in zip(items, answers):
        p = parse_json(ans)
        g = it["gold"]
        r = {"text": it["text"], "answer": ans, "valid_json": p is not None,
             "category_gold": g["category"], "currency_gold": g["currency"] or "not stated", "gold_flag": g["flag"]}
        for f in FIELDS7:
            r[f] = p is not None and same(f, p.get(f), g[f])
        r["reason_present"] = p is not None and isinstance(p.get("reason"), str) and len(p["reason"].strip()) > 0
        r["all_7_fields"] = all(r[f] for f in FIELDS7)
        r["pred_flag"] = str((p or {}).get("flag") or "invalid").strip().lower()
        rows.append(r)
    return pd.DataFrame(rows)

def summarize(df):
    s = df[["valid_json"] + FIELDS7 + ["reason_present", "all_7_fields"]].mean().round(3)
    s["flag_macro_f1"] = round(f1_score(df.gold_flag, df.pred_flag, average="macro", labels=FLAGS, zero_division=0), 3)
    return s

def report(df, name):
    print(f"===== {name}: {len(df)} expenses =====")
    print(summarize(df).to_string())
    print("\nAll 7 fields correct, by category:\n", df.groupby("category_gold")["all_7_fields"].mean().round(3).to_string())
    print("\nAll 7 fields correct, by correct flag:\n", df.groupby("gold_flag")["all_7_fields"].mean().round(3).to_string())
    print("\nAll 7 fields correct, by currency:\n", df.groupby("currency_gold")["all_7_fields"].mean().round(3).to_string())
    cm = pd.crosstab(df.gold_flag, df.pred_flag).reindex(index=FLAGS, columns=FLAGS + ["invalid"], fill_value=0)
    print("\nFlag confusion matrix (rows = correct flag, columns = model's flag):\n", cm.to_string())

def show_mistakes(df, n=10):
    wrong = df[~df.all_7_fields]
    print(f"\n{len(wrong)} of {len(df)} have at least one wrong field. First {min(n, len(wrong))}:\n")
    for _, r in wrong.head(n).iterrows():
        print("TEXT :", r.text)
        print("MODEL:", r.answer.strip()[:300])
        print("WRONG:", [f for f in FIELDS7 if not r[f]], "| correct flag:", r.gold_flag, "\n")

## 6. Test 1: the 1,000 held-out rows
Training split the 5,000 rows randomly 80/20 with `seed=42`. Re-creating the split with the same seed gives the same 1,000 rows.

**Caveat to say out loud:** these rows come from the same generator as the training rows, with the same vendors and templates. A high score here shows the model learned the task on this kind of data. It does not show it handles new vendors or real expense text; Test 3 is the closer check.

In [7]:
from datasets import load_dataset

raw = load_dataset("csv", data_files=CSV_PATH, split="train")
raw = raw.select(range(5000))
heldout = raw.train_test_split(test_size=0.2, shuffle=True, seed=42)["test"]    # same call as in training
print(len(raw), "rows in the file;", len(heldout), "held out")

def clean(v):
    # empty CSV cells arrive as None / NaN -> treat as "not stated"
    return None if v is None or (isinstance(v, float) and math.isnan(v)) or v == "" else v

def csv_gold(r):
    num = lambda v: None if clean(v) is None else float(v)
    txt = lambda v: None if clean(v) is None else str(v)
    return {"amount": num(r["Amount"]), "currency": txt(r["Currency"]), "category": txt(r["Category"]),
            "policy_limit_amount": num(r["Policy_limit_amount"]),
            "reimbursement_allowed": txt(r["Reimbursement_allowed"]), "receipt_provided": txt(r["Receipt_provided"]),
            "flag": txt(r["Flag"])}

items = [{"text": r["Text"], "gold": csv_gold(r)} for r in heldout]
if N_HELDOUT:
    items = items[:N_HELDOUT]
print("Example:", items[0]["text"], "->", items[0]["gold"])

Generating train split: 0 examples [00:00, ? examples/s]

5000 rows in the file; 1000 held out
Example: Holiday Inn Birmingham for the workshop, 7 nts £796.61 -> {'amount': 796.61, 'currency': 'GBP', 'category': 'Lodging', 'policy_limit_amount': 1400.0, 'reimbursement_allowed': 'Yes', 'receipt_provided': 'Yes', 'flag': 'none'}


In [8]:
answers_ft = run_model(model, tok, [it["text"] for it in items])
df_ft = score(items, answers_ft)
report(df_ft, "Fine-tuned model on held-out rows")
show_mistakes(df_ft)

  1000/1000 answered, 447s
===== Fine-tuned model on held-out rows: 1000 expenses =====
valid_json               1.000
amount                   1.000
currency                 1.000
category                 0.998
policy_limit_amount      0.991
reimbursement_allowed    1.000
receipt_provided         1.000
flag                     0.996
reason_present           1.000
all_7_fields             0.987
flag_macro_f1            0.996

All 7 fields correct, by category:
 category_gold
Ground Transport    1.000
Lodging             1.000
Meals               0.933
Office Supplies     1.000
Other               0.984
Software            0.975
Training            1.000
Travel              1.000

All 7 fields correct, by correct flag:
 gold_flag
missing_info        0.992
none                0.971
not_reimbursable    1.000
over_limit          0.985

All 7 fields correct, by currency:
 currency_gold
CAD           0.988
EUR           0.974
GBP           0.988
INR           0.989
USD           0.989
not st

## 7. Test 2: baseline (the untrained model)
Same prompt, same rows. This is the number fine-tuning has to beat. The fine-tuned model is freed from memory first, so both fit on the GPU.

In [9]:
n_base = min(N_BASELINE, len(items))
base_items = items[:n_base]

del model
gc.collect(); torch.cuda.empty_cache()

base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto", dtype=torch.float16).eval()
answers_base = run_model(base_model, base_tok, [it["text"] for it in base_items])
df_base = score(base_items, answers_base)
report(df_base, "UNTRAINED Qwen2.5-1.5B-Instruct (baseline), same prompt")
print("\nSample baseline answer:\n", answers_base[0][:400])

del base_model
gc.collect(); torch.cuda.empty_cache()

# reload the fine-tuned model for the remaining tests
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype=torch.float16).eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  200/200 answered, 82s
===== UNTRAINED Qwen2.5-1.5B-Instruct (baseline), same prompt: 200 expenses =====
valid_json               0.955
amount                   0.795
currency                 0.825
category                 0.635
policy_limit_amount      0.135
reimbursement_allowed    0.005
receipt_provided         0.910
flag                     0.255
reason_present           0.750
all_7_fields             0.000
flag_macro_f1            0.216

All 7 fields correct, by category:
 category_gold
Ground Transport    0.0
Lodging             0.0
Meals               0.0
Office Supplies     0.0
Other               0.0
Software            0.0
Training            0.0
Travel              0.0

All 7 fields correct, by correct flag:
 gold_flag
missing_info        0.0
none                0.0
not_reimbursable    0.0
over_limit          0.0

All 7 fields correct, by currency:
 currency_gold
CAD           0.0
EUR           0.0
GBP           0.0
INR           0.0
USD           0.0
not stated    0.0

Fla

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
# Side by side on the SAME rows (the baseline ran on the first n_base rows)
same_rows = df_ft.iloc[:n_base]
comparison = pd.DataFrame({"untrained": summarize(df_base), "fine_tuned": summarize(same_rows)})
print(f"Both scored on the same {n_base} held-out rows:\n")
print(comparison.to_string())

Both scored on the same 200 held-out rows:

                       untrained  fine_tuned
valid_json                 0.955       1.000
amount                     0.795       1.000
currency                   0.825       1.000
category                   0.635       1.000
policy_limit_amount        0.135       0.995
reimbursement_allowed      0.005       1.000
receipt_provided           0.910       1.000
flag                       0.255       1.000
reason_present             0.750       1.000
all_7_fields               0.000       0.995
flag_macro_f1              0.216       1.000


## 8. Test 3: fresh cases the model has never seen
26 new expense lines, written with vendors and wording that do not appear in the dataset generator.
Their right answers are **computed from the policy table** in `policy_gold` below (the same rules used to label the dataset), so no arithmetic is typed by hand.

Each case is: `(text, category, currency, amount, attendees, nights, receipt_present, not_reimbursable)`.

In [11]:
LIMITS = {   # USD, EUR, GBP, INR, CAD
    "Meals": (75, 70, 60, 6000, 100), "Lodging": (250, 230, 200, 20000, 340),
    "Ground Transport": (100, 90, 80, 8000, 135), "Travel": (800, 750, 650, 64000, 1100),
    "Software": (200, 180, 160, 16000, 270), "Office Supplies": (150, 140, 120, 12000, 200),
    "Training": (1000, 900, 800, 80000, 1350), "Other": (100, 90, 80, 8000, 135)}
CURS = ["USD", "EUR", "GBP", "INR", "CAD"]
RECEIPT_ABOVE = dict(zip(CURS, (25, 25, 20, 2000, 35)))

def policy_gold(cat, cur, amount, attendees=None, nights=None, receipt=True, nr=False):
    # Right answer for one expense, from the policy: not_reimbursable > missing_info > over_limit > none
    base = {"amount": amount, "currency": cur if amount is not None else None, "category": cat,
            "receipt_provided": "Yes" if receipt else "No"}
    if nr:
        return base | {"policy_limit_amount": 0, "reimbursement_allowed": "No", "flag": "not_reimbursable"}
    unit = LIMITS[cat][CURS.index(cur)]
    missing, total = amount is None, None
    if cat == "Meals":
        missing |= attendees is None
        total = None if attendees is None else unit * attendees
    elif cat == "Lodging":
        missing |= nights is None
        total = None if nights is None else unit * nights
    else:
        total = unit
    if amount is None:
        total = None
    if not receipt and (amount is None or amount > RECEIPT_ABOVE[cur]):
        missing = True
    flag = "missing_info" if missing else ("over_limit" if amount > total else "none")
    return base | {"policy_limit_amount": total, "reimbursement_allowed": "Yes", "flag": flag}

FRESH = [
 ("Took a Gojek from the hotel to the client office, 18.50 USD", "Ground Transport", "USD", 18.50, None, None, True, False),
 ("Careem ride to the conference venue, 140 usd", "Ground Transport", "USD", 140, None, None, True, False),
 ("Lunch for the whole design team at Bella Napoli - 7 of us - EUR 420", "Meals", "EUR", 420, 7, None, True, False),
 ("Dinner with a client at Gordon Ramsay Steak, party of 3, £265", "Meals", "GBP", 265, 3, None, True, False),
 ("Team dinner at Jaipur Palace, 6 ppl, Rs 31,500, lost the receipt", "Meals", "INR", 31500, 6, None, False, False),
 ("Sushi Zanmai lunch, just me, C$ 62", "Meals", "CAD", 62, 1, None, True, False),
 ("Ritz-Carlton Chicago, 2 nights, $1,150", "Lodging", "USD", 1150, None, 2, True, False),
 ("Lemon Tree Hotel Pune - 3 nights - Rs 24,900", "Lodging", "INR", 24900, None, 3, True, False),
 ("Pullman Berlin for the trade fair, EUR 780, receipt attached", "Lodging", "EUR", 780, None, None, True, False),
 ("Kimpton London, 2 nts, GBP 310", "Lodging", "GBP", 310, None, 2, True, False),
 ("Qantas flight Sydney to Singapore for the offsite, 640 USD", "Travel", "USD", 640, None, None, True, False),
 ("Ryanair return flights to Dublin, EUR 1,020", "Travel", "EUR", 1020, None, None, True, False),
 ("Vistara Delhi to Mumbai ticket Rs 71,000", "Travel", "INR", 71000, None, None, True, False),
 ("Sentry annual team plan, 310 USD", "Software", "USD", 310, None, None, True, False),
 ("Zapier subscription renewal for the ops team, GBP 85", "Software", "GBP", 85, None, None, True, False),
 ("Airtable seats, CAD 190, no receipt", "Software", "CAD", 190, None, None, False, False),
 ("Lyreco order: toner and paper reams EUR 96.40", "Office Supplies", "EUR", 96.40, None, None, True, False),
 ("Daiso - stationery for the workshop, Rs 1,350", "Office Supplies", "INR", 1350, None, None, True, False),
 ("Codecademy Pro annual plan for me, USD 240", "Training", "USD", 240, None, None, True, False),
 ("Harvard Online negotiation course GBP 1,450", "Training", "GBP", 1450, None, None, True, False),
 ("Shipping a client package with Aramex, 38 USD", "Other", "USD", 38, None, None, True, False),
 ("Notary charges for the vendor agreement, CAD 160", "Other", "CAD", 160, None, None, True, False),
 ("Bought drinks for everyone at the offsite party, bar tab 94 USD", "Meals", "USD", 94, None, None, True, True),
 ("Parking fine near the client site, 45 USD", "Ground Transport", "USD", 45, None, None, True, True),
 ("Haircut and spa day while travelling, GBP 130", "Other", "GBP", 130, None, None, True, True),
 ("Business class upgrade on Qantas, 520 USD", "Travel", "USD", 520, None, None, True, True),
]
fresh_items = [{"text": t, "gold": policy_gold(cat, cur, amt, att, nts, rc, nr)}
               for (t, cat, cur, amt, att, nts, rc, nr) in FRESH]
print(len(fresh_items), "fresh cases. Right flags:", pd.Series([i["gold"]["flag"] for i in fresh_items]).value_counts().to_dict())

26 fresh cases. Right flags: {'none': 11, 'over_limit': 8, 'not_reimbursable': 4, 'missing_info': 3}


In [12]:
answers_fresh = run_model(model, tok, [it["text"] for it in fresh_items])
df_fresh = score(fresh_items, answers_fresh)
report(df_fresh, "Fine-tuned model on FRESH cases (unseen vendors and wording)")
show_mistakes(df_fresh, n=26)

  26/26 answered, 12s
===== Fine-tuned model on FRESH cases (unseen vendors and wording): 26 expenses =====
valid_json               1.000
amount                   1.000
currency                 1.000
category                 0.962
policy_limit_amount      0.962
reimbursement_allowed    1.000
receipt_provided         1.000
flag                     0.962
reason_present           1.000
all_7_fields             0.962
flag_macro_f1            0.973

All 7 fields correct, by category:
 category_gold
Ground Transport    1.0
Lodging             1.0
Meals               1.0
Office Supplies     1.0
Other               1.0
Software            1.0
Training            0.5
Travel              1.0

All 7 fields correct, by correct flag:
 gold_flag
missing_info        1.000
none                0.909
not_reimbursable    1.000
over_limit          1.000

All 7 fields correct, by currency:
 currency_gold
CAD    1.0
EUR    1.0
GBP    1.0
INR    1.0
USD    0.9

Flag confusion matrix (rows = correct flag, co

## 9. Test 4: edge cases (looked at, not scored)
These are inputs the training data never covered. There is no single right answer, so read the output and note what the model does.
A sensible model flags what is missing (`missing_info`, and `null` for what the text does not state) instead of making something up.

In [13]:
EDGE = [
    "Yesterday I spent 12000 for flight ticket",     # amount but no currency
    "Taxi 500",                                      # no currency, no context
    "Lunch with the team, receipt attached",         # no amount, no attendees
    "Dinner 3 people 90 euros",                      # plain wording, in range
    "hotel 4 nights 900",                            # no currency
    "Cab from airport 65 dollars no bill",           # missing receipt, amount above the receipt threshold
    "Conference ticket 1,500 CAD for me and my wife",# family member: should this be personal?
    "asdf 123 !!!",                                  # nonsense
]
for text, ans in zip(EDGE, run_model(model, tok, EDGE)):
    print("INPUT :", text)
    print("OUTPUT:", (parse_json(ans) or ans))
    print()

  8/8 answered, 5s
INPUT : Yesterday I spent 12000 for flight ticket
OUTPUT: {'amount': 12000.0, 'currency': 'INR', 'category': 'Travel', 'policy_limit_amount': 64000.0, 'reimbursement_allowed': 'Yes', 'receipt_provided': 'Yes', 'flag': 'none', 'reason': 'INR 12,000.00 is within the INR 64,000 Travel limit.'}

INPUT : Taxi 500
OUTPUT: {'amount': 500.0, 'currency': 'INR', 'category': 'Ground Transport', 'policy_limit_amount': 8000.0, 'reimbursement_allowed': 'Yes', 'receipt_provided': 'Yes', 'flag': 'none', 'reason': 'INR 500.00 is within the INR 8,000 Ground Transport limit.'}

INPUT : Lunch with the team, receipt attached
OUTPUT: {'amount': None, 'currency': None, 'category': 'Meals', 'policy_limit_amount': None, 'reimbursement_allowed': 'Yes', 'receipt_provided': 'Yes', 'flag': 'missing_info', 'reason': 'The amount is not stated.'}

INPUT : Dinner 3 people 90 euros
OUTPUT: {'amount': 90.0, 'currency': 'EUR', 'category': 'Meals', 'policy_limit_amount': 210.0, 'reimbursement_allowed': 

## 10. Summary table
Fill the numbers into your notes and slides. Always say which test each number comes from.

In [14]:
summary = pd.DataFrame({
    "baseline (untrained), held-out rows": summarize(df_base),
    "fine-tuned, same rows": summarize(df_ft.iloc[:n_base]),
    "fine-tuned, all held-out rows": summarize(df_ft),
    "fine-tuned, fresh cases": summarize(df_fresh),
})
print(summary.to_string())
summary.to_csv("evaluation_summary.csv")
df_ft.to_csv("heldout_results.csv", index=False)
df_fresh.to_csv("fresh_results.csv", index=False)
print("\nSaved evaluation_summary.csv, heldout_results.csv, fresh_results.csv (download them from the file panel).")

                       baseline (untrained), held-out rows  fine-tuned, same rows  fine-tuned, all held-out rows  fine-tuned, fresh cases
valid_json                                           0.955                  1.000                          1.000                    1.000
amount                                               0.795                  1.000                          1.000                    1.000
currency                                             0.825                  1.000                          1.000                    1.000
category                                             0.635                  1.000                          0.998                    0.962
policy_limit_amount                                  0.135                  0.995                          0.991                    0.962
reimbursement_allowed                                0.005                  1.000                          1.000                    1.000
receipt_provided                  